In [3]:
!pip install -U -q chromadb langchain langchain-community langchain-text-splitters sentence-transformers pandas

In [4]:
import os
import pandas as pd
import shutil
from langchain_community.document_loaders import DataFrameLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# تحديد مسارات Kaggle القياسية للمخرجات
CHROMA_DB_PATH = "/kaggle/working/chroma_db_laws"
ZIP_OUTPUT_PATH = "/kaggle/working/chroma_db_laws_backup"

def find_csv_file(filename="processed_laws.csv"):
    """دالة مساعدة للبحث عن ملف الـ CSV في بيئة Kaggle"""
    if os.path.exists(f"/kaggle/working/{filename}"):
        return f"/kaggle/working/{filename}"
    
    for root, dirs, files in os.walk("/kaggle/input"):
        if filename in files:
            return os.path.join(root, filename)
            
    return None

def main():
    print("⏳ جاري البحث عن ملف القوانين المعالجة (CSV)...")
    csv_path = find_csv_file()
    
    if not csv_path:
        print("❌ لم يتم العثور على الملف: processed_laws.csv")
        print("يرجى التأكد من تشغيل كود معالجة البيانات أولاً أو رفع الملف كـ Dataset.")
        return

    print(f"📍 تم العثور على الملف في: {csv_path}")
    
    df = pd.read_csv(csv_path)
    
    loader = DataFrameLoader(df, page_content_column="text")
    documents = loader.load()
    print(f"✅ تم تحميل {len(documents)} مادة قانونية مع بياناتها الوصفية.")

    print("⏳ جاري تطبيق التقسيم الاحتياطي (Fallback Chunking)...")
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1500,
        chunk_overlap=150,
        separators=["\n\n", "\n", " ", ""],
        is_separator_regex=False
    )
    splitted_docs = text_splitter.split_documents(documents)
    print(f"✅ تم تجهيز {len(splitted_docs)} مقطعاً نهائياً.")

    print("⏳ جاري تحميل نموذج التضمين (Embedding Model)...")
    embedding_model = HuggingFaceEmbeddings(
        model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
    )

    print("⏳ جاري بناء متجر المتجهات ChromaDB... (يرجى الانتظار)")
    if os.path.exists(CHROMA_DB_PATH):
        shutil.rmtree(CHROMA_DB_PATH)
        
    vectordb = Chroma.from_documents(
        documents=splitted_docs,
        embedding=embedding_model,
        persist_directory=CHROMA_DB_PATH
    )
    
    vectordb.persist()
    print("🎉 تم بناء قاعدة البيانات المتجهة بنجاح!")
    
    shutil.make_archive(ZIP_OUTPUT_PATH, 'zip', CHROMA_DB_PATH)
    print(f"⬇️ الملف جاهز للتحميل من قسم Output في Kaggle باسم: chroma_db_laws_backup.zip")

if __name__ == "__main__":
    main()

⏳ جاري البحث عن ملف القوانين المعالجة (CSV)...
📍 تم العثور على الملف في: /kaggle/input/datasets/maherghanem/processed-laws/processed_laws.csv
✅ تم تحميل 2839 مادة قانونية مع بياناتها الوصفية.
⏳ جاري تطبيق التقسيم الاحتياطي (Fallback Chunking)...


/tmp/ipykernel_57/2311039644.py:52: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


✅ تم تجهيز 4157 مقطعاً نهائياً.
⏳ جاري تحميل نموذج التضمين (Embedding Model)...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

⏳ جاري بناء متجر المتجهات ChromaDB... (يرجى الانتظار)


/tmp/ipykernel_57/2311039644.py:66: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectordb.persist()


🎉 تم بناء قاعدة البيانات المتجهة بنجاح!
⬇️ الملف جاهز للتحميل من قسم Output في Kaggle باسم: chroma_db_laws_backup.zip
